In [1]:
import re
import emoji
import joblib

from pythainlp.tokenize import word_tokenize
from scipy.sparse import hstack

In [2]:
def clean_text(text):
    text = str(text)
    text = re.sub(r'&#\d+;', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' <URL> ', text)
    text = re.sub(r'@\S+', ' <USER> ', text)
    text = emoji.replace_emoji(text, replace=' <EMOJI> ')
    text = re.sub(r'[^ก-๙a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def thai_tokenizer(text):
    return word_tokenize(text, engine="newmm")

In [3]:
logreg_model = joblib.load("logistic_regression_model.pkl")
tfidf_word = joblib.load("tfidf_word_logreg.pkl")
tfidf_char = joblib.load("tfidf_char_logreg.pkl")
le = joblib.load("label_encoder.pkl")

In [4]:
def predict_logreg(text):
    # clean input
    text = clean_text(text)

    # TF-IDF features
    Xw = tfidf_word.transform([text])
    Xc = tfidf_char.transform([text])

    # combine features
    X = hstack([Xw, Xc])

    # predict label
    pred = logreg_model.predict(X)

    # convert to sentiment name
    return le.inverse_transform(pred)[0]

In [5]:
predict_logreg("รถไฟฟ้าช้ามาก เดือดร้อนสุดๆ")

'negative'

In [6]:
predict_logreg("โครงการนี้ช่วยประชาชนได้ดีมาก")

'negative'

In [7]:
predict_logreg("วันนี้อากาศร้อน")

'neutral'

In [8]:
predict_logreg("นโยบายนี้สร้างปัญหาให้ประชาชน")

'neutral'